# Section 4: Generalization Pipeline

## Pipeline Overview
1. Load test data
2. Preprocess using fitted scaler and selected features
3. Predict cluster IDs
4. Route to appropriate subgroup models
5. Enforce <20% bankrupt constraint
6. Generate submission CSV

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
from sklearn.preprocessing import StandardScaler, RobustScaler

# Required libraries for model loading
from sklearn.ensemble import (
    RandomForestClassifier, 
    ExtraTreesClassifier, 
    HistGradientBoostingClassifier, 
    StackingClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

RANDOM_STATE = 42

## Configuration

In [2]:
# --- Configuration ---
TEST_DATA_PATH = "test_data.csv"
OUTPUT_FILE = "submission.csv"
LIMIT_PCT = 0.20  # Maximum allowed percentage of predicted bankrupts

# Model selection for Cluster 2
# Options: 'C', 'D', or 'ensemble'
CLUSTER2_MODEL = 'D'  # Use alternative model by default

# --- Model Paths ---
PREPROCESS_JOBLIB = "preprocess_for_clustering.joblib"
FEATURES_JOBLIB = "top_features_for_clustering.joblib"
CLUSTER_ID_MODEL = "cluster_4_model.joblib"

# Subgroup Models
MODEL_A_PATH = "cluster0_stacking_PNH.joblib"  # Cluster 0
MODEL_B_PATH = "cluster4_stacking_B.joblib"    # Cluster 4
MODEL_C_PATH = "cluster2_stacking_C.joblib"    # Cluster 2 (Model C)
MODEL_D_PATH = "cluster2_stacking_D.joblib"    # Cluster 2 (Alternative)

## Load Models and Preprocessing

In [3]:
def load_pipeline_from_joblib(path, key=None):
    """Robustly loads a joblib object (pipeline, model, or package dict)."""
    if not os.path.exists(path):
        print(f"FATAL ERROR: Model file {path} not found.")
        sys.exit(1)
    
    obj = joblib.load(path)
    
    if isinstance(obj, dict) and key:
        return obj.get(key)
    
    return obj

# 1. Load Scaler & Feature Names
preprocess_pkg = load_pipeline_from_joblib(PREPROCESS_JOBLIB)
scaler = preprocess_pkg.get("scaler") if isinstance(preprocess_pkg, dict) else preprocess_pkg
feature_names = load_pipeline_from_joblib(FEATURES_JOBLIB)
print(f"Loaded preprocessing: {len(feature_names)} features")

# 2. Load Cluster-ID Model
clf_cluster = load_pipeline_from_joblib(CLUSTER_ID_MODEL, key="model")
print("Loaded cluster-ID classifier")

# 3. Load Subgroup Models
model_0 = load_pipeline_from_joblib(MODEL_A_PATH, key="pipeline")
print("Loaded Cluster 0 model")

model_4 = load_pipeline_from_joblib(MODEL_B_PATH, key="pipeline")
print("Loaded Cluster 4 model")

# Load Cluster 2 model(s) based on configuration
model_2 = None
model_2_alt = None

if CLUSTER2_MODEL in ['C', 'ensemble']:
    model_2 = load_pipeline_from_joblib(MODEL_C_PATH, key="pipeline")
    print("Loaded Cluster 2 model (C)")

if CLUSTER2_MODEL in ['D', 'ensemble']:
    model_2_alt = load_pipeline_from_joblib(MODEL_D_PATH, key="pipeline")
    print("Loaded Cluster 2 alternative model (D)")

# Set primary model for cluster 2
if CLUSTER2_MODEL == 'D':
    model_2 = model_2_alt
    model_2_alt = None

print("\nAll models loaded successfully.")

Loaded preprocessing: 40 features
Loaded cluster-ID classifier
Loaded Cluster 0 model
Loaded Cluster 4 model
Loaded Cluster 2 alternative model (D)

All models loaded successfully.


## Load and Preprocess Test Data

In [4]:
# Load test data
if not os.path.exists(TEST_DATA_PATH):
    print(f"FATAL ERROR: Test data file '{TEST_DATA_PATH}' not found.")
    print("Please ensure the test file is in the same directory before submission.")
    sys.exit(1)

df_test = pd.read_csv(TEST_DATA_PATH)
print(f"Test data loaded. Shape: {df_test.shape}")

# Validate Index column
if 'Index' not in df_test.columns:
    print("FATAL ERROR: 'Index' column not found in test data.")
    sys.exit(1)

# Apply preprocessing
try:
    X_test_raw = df_test[feature_names].copy()
    X_test_scaled = scaler.transform(X_test_raw)
    print("Test data scaled successfully.")
except KeyError as e:
    print(f"FATAL ERROR: Test CSV missing required feature: {e}")
    sys.exit(1)

Test data loaded. Shape: (1012, 96)
Test data scaled successfully.


## Predict Cluster IDs

In [5]:
# Predict cluster assignments
test_clusters = clf_cluster.predict(X_test_scaled)
df_test['predicted_cluster'] = test_clusters

print("Cluster Assignments on Test Data:")
print(df_test['predicted_cluster'].value_counts().sort_index())

Cluster Assignments on Test Data:
predicted_cluster
0    126
1    135
2    351
4    400
Name: count, dtype: int64


## Route to Subgroup Models and Predict

In [6]:
# Initialize prediction storage
test_probs = np.zeros(len(df_test))

print("\nRouting Test Rows to Subgroup Models")

for cluster_id in sorted(df_test['predicted_cluster'].unique()):
    mask = df_test['predicted_cluster'] == cluster_id
    X_subset = df_test.loc[mask, feature_names]  # Pass unscaled features to Pipeline
    count = mask.sum()
    
    if count == 0:
        continue
    
    print(f"\nProcessing Cluster {cluster_id}: {count} rows")
    
    if cluster_id == 0 and model_0:
        # Cluster 0 model
        test_probs[mask] = model_0.predict_proba(X_subset)[:, 1]
        
    elif cluster_id == 4 and model_4:
        # Cluster 4 model
        test_probs[mask] = model_4.predict_proba(X_subset)[:, 1]
        
    elif cluster_id == 2 and model_2:
        # Cluster 2 model(s)
        if CLUSTER2_MODEL == 'ensemble' and model_2_alt is not None:
            # Ensemble: average predictions from both models
            probs_1 = model_2.predict_proba(X_subset)[:, 1]
            probs_2 = model_2_alt.predict_proba(X_subset)[:, 1]
            test_probs[mask] = (probs_1 + probs_2) / 2
            print("Using Cluster 2 ensemble (C+D)")
        else:
            # Single model
            test_probs[mask] = model_2.predict_proba(X_subset)[:, 1]
            print(f"Using Cluster 2 model ({CLUSTER2_MODEL})")
            
    elif cluster_id in [1, 5]:
        # Constant 0: Clusters with no bankruptcies in training
        test_probs[mask] = 0.0
        print("Constant 0 (Safe cluster)")
        
    elif cluster_id in [3, 6]:
        # Constant 1: High-risk tiny clusters
        test_probs[mask] = 1.0
        print("Constant 1 (High-risk cluster)")
        
    else:
        # Unknown cluster - default to 0
        test_probs[mask] = 0.0
        print("Unknown cluster - defaulting to 0.0")

df_test['pred_prob'] = test_probs
print(f"\nPredictions generated for all {len(df_test)} test samples")


Routing Test Rows to Subgroup Models

Processing Cluster 0: 126 rows



Processing Cluster 1: 135 rows
Constant 0 (Safe cluster)

Processing Cluster 2: 351 rows
Using Cluster 2 model (D)

Processing Cluster 4: 400 rows

Predictions generated for all 1012 test samples


## Post-Processing: Enforce <20% Constraint

In [7]:
# Initial predictions with 0.5 threshold
final_preds = (test_probs > 0.5).astype(int)
num_positive = final_preds.sum()
pct_positive = num_positive / len(df_test)

print("\n--- Post-Processing: <20% Constraint Enforcement ---")
print(f"Initial predictions (threshold=0.5): {num_positive} bankrupts ({pct_positive:.2%})")

# Check and enforce constraint
if pct_positive > LIMIT_PCT:
    print(f"VIOLATION: {pct_positive:.2%} > {LIMIT_PCT:.0%}")
    print("Applying constraint enforcement...")
    
    # Use 19% to have safety margin
    target_count = int(len(df_test) * (LIMIT_PCT - 0.01))
    print(f"Target: {target_count} bankrupts ({(LIMIT_PCT - 0.01):.0%})")
    
    # Sort by probability and select top K
    sorted_indices = np.argsort(test_probs)[::-1]
    
    # Reset predictions
    new_preds = np.zeros(len(df_test), dtype=int)
    new_preds[sorted_indices[:target_count]] = 1
    
    final_preds = new_preds
    print(f"Adjusted predictions: {final_preds.sum()} bankrupts ({final_preds.sum()/len(df_test):.2%})")
else:
    print(f"Constraint satisfied: {pct_positive:.2%} < {LIMIT_PCT:.0%}")

df_test['Bankrupt?'] = final_preds


--- Post-Processing: <20% Constraint Enforcement ---
Initial predictions (threshold=0.5): 116 bankrupts (11.46%)
Constraint satisfied: 11.46% < 20%


## Generate Submission File

In [8]:
# Create submission
submission = df_test[['Index', 'Bankrupt?']].copy()

# Final validation
assert submission.shape[0] == df_test.shape[0], "Row count mismatch"
assert submission['Bankrupt?'].isnull().sum() == 0, "Contains null predictions"
assert submission['Bankrupt?'].mean() < LIMIT_PCT, f"Constraint violated: {submission['Bankrupt?'].mean():.2%}"

# Save to CSV
submission.to_csv(OUTPUT_FILE, index=False)